<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/deepseek2gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import sys
from google.colab import userdata
from huggingface_hub import HfApi

# Target credentials and repo variables
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'

In [6]:
!ls -lh /content/drive/MyDrive/topogpt/

total 255G
drwx------ 2 root root 4.0K Nov  7 21:14 base_backbone
drwx------ 2 root root 4.0K Nov  1 12:22 base_model_70b
drwx------ 2 root root 4.0K Nov  1 12:22 checkpoints
drwx------ 2 root root 4.0K Nov  7 06:20 deepseek_r1_70b_topo_governed_merged
-rw------- 1 root root    0 Nov  7 22:04 deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf
drwx------ 2 root root 4.0K Oct 26 15:10 gguf
drwx------ 2 root root 4.0K Nov  7 06:33 gguf_scratch
drwx------ 2 root root 4.0K Oct 19 19:52 hf_cache
drwx------ 2 root root 4.0K Nov  2 11:04 llama.cpp
drwx------ 2 root root 4.0K Nov  7 18:57 merged_deepseek_topo_hf
drwx------ 2 root root 4.0K Nov  7 05:10 merged_model
drwx------ 2 root root 4.0K Nov  6 20:16 merged_sarvam_topo_hf
drwx------ 2 root root 4.0K Nov  3 04:26 mixtral_8x22b_bf16_hf
-rw------- 1 root root  80G Nov  4 06:49 mixtral_8x22b_q4_k_m.gguf
-rw------- 1 root root  37G Nov  3 18:55 mixtral_8x22b_q8_0-00001-of-00004.gguf
-rw------- 1 root root  38G Nov  3 21:46 mixtral_8x22b_q8_0

In [ ]:
#!rm -rf /content/drive/MyDrive/topogpt/base_backbone
#!rm -rf /content/drive/MyDrive/topogpt/deepseek_r1_70b_topo_governed_merged
#!rm -rf /content/drive/MyDrive/topogpt/topo_cbp_trained_weights.safetensors

In [3]:
!ls -lh /content/drive/MyDrive/topogpt/topo_cbp_trained_weights.safetensors

-rw------- 1 root root 4.0G Nov  7 02:57 /content/drive/MyDrive/topogpt/topo_cbp_trained_weights.safetensors


In [4]:
from huggingface_hub import snapshot_download

print("=" * 70)
print("📥 DOWNLOADING DEEPSEEK-R1-DISTILL-LLAMA-70B TO LOCAL NVME")
print("=" * 70)

snapshot_download(
    repo_id="deepseek-ai/DeepSeek-R1-Distill-Llama-70B",
    local_dir="/content/local_backbone",
    local_dir_use_symlinks=False
)

print("✅ Download complete on local NVMe!")

📥 DOWNLOADING DEEPSEEK-R1-DISTILL-LLAMA-70B TO LOCAL NVME


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

✅ Download complete on local NVMe!


In [12]:
!cp -pr /content/drive/MyDrive/topogpt/topo_cbp_trained_weights.safetensors /content/local_backbone/

In [15]:
!ls -lth /content/local_backbone/

total 136G
-rw-r--r-- 1 root root  59K Nov  8 01:38 model.safetensors.index.json
-rw------- 1 root root 9.9G Nov  8 01:38 model-00017-of-000017.safetensors
-rw-r--r-- 1 root root 8.1G Nov  7 23:39 model-00016-of-000017.safetensors
-rw-r--r-- 1 root root 7.9G Nov  7 23:39 model-00015-of-000017.safetensors
-rw-r--r-- 1 root root 8.1G Nov  7 23:39 model-00014-of-000017.safetensors
-rw-r--r-- 1 root root 7.9G Nov  7 23:39 model-00013-of-000017.safetensors
-rw-r--r-- 1 root root 8.1G Nov  7 23:39 model-00012-of-000017.safetensors
-rw-r--r-- 1 root root 7.9G Nov  7 23:39 model-00011-of-000017.safetensors
-rw-r--r-- 1 root root 3.0K Nov  7 23:38 tokenizer_config.json
-rw-r--r-- 1 root root 8.7M Nov  7 23:38 tokenizer.json
-rw-r--r-- 1 root root 8.1G Nov  7 23:38 model-00010-of-000017.safetensors
-rw-r--r-- 1 root root 7.9G Nov  7 23:36 model-00009-of-000017.safetensors
-rw-r--r-- 1 root root 8.1G Nov  7 23:36 model-00002-of-000017.safetensors
-rw-r--r-- 1 root root 8.1G Nov  7 23:36 model-000

## gguf

In [16]:
import os
import json
import torch
import subprocess
from safetensors.torch import load_file, save_file

LOCAL_WORK_DIR = "/content/local_backbone"
TOPO_PATH = os.path.join(LOCAL_WORK_DIR, "topo_cbp_trained_weights.safetensors")
TARGET_SHARD = "model-00017-of-000017.safetensors"
shard_path = os.path.join(LOCAL_WORK_DIR, TARGET_SHARD)
bin_path = shard_path.replace(".safetensors", ".bin")
LLAMA_CPP_DIR = "/content/drive/MyDrive/topogpt/llama.cpp"
LOCAL_GGUF = "/content/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"

print("=" * 70)
print("⚙️ ZERO-DRIFT TOPO-CBP GGUF GENERATION")
print("=" * 70)

# 1. Clean up any accidental .bin format and ensure shard 17 is a valid safetensors file
if os.path.exists(bin_path):
    print("Converting stray .bin shard back to safetensors...")
    state_dict = torch.load(bin_path, map_location="cpu")
    save_file(state_dict, shard_path)
    os.remove(bin_path)

# 2. Fix index.json to point strictly to the safetensors extension
index_json_path = os.path.join(LOCAL_WORK_DIR, "model.safetensors.index.json")
with open(index_json_path, 'r') as f:
    index_data = json.load(f)

weight_map = index_data.get("weight_map", {})
for k, v in weight_map.items():
    if v != TARGET_SHARD:
        # Normalize any stray .bin references in the index map
        if "model-00017-of-000017" in v:
            weight_map[k] = TARGET_SHARD

with open(index_json_path, 'w') as f:
    json.dump(index_data, f, indent=2)

# 3. Load zero-drift weights and inject into shard 17 using native safetensors
print("Loading zero-drift Topo-CBP weights...")
topo_weights = load_file(TOPO_PATH, device="cpu")
keys_to_patch = [k for k, v in weight_map.items() if v == TARGET_SHARD and k in topo_weights]
print(f"Patching {len(keys_to_patch)} tensors into {TARGET_SHARD}...")

shard_state = load_file(shard_path, device="cpu")
for key in keys_to_patch:
    shard_state[key] = topo_weights[key].detach().cpu().contiguous()

save_file(shard_state, shard_path)
print("✅ Shard patched successfully in safetensors format.")

# 4. Run llama.cpp conversion to create the final Q8_0 GGUF model
print("Running llama.cpp conversion...")
convert_cmd = [
    "python", os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
    LOCAL_WORK_DIR,
    "--outfile", LOCAL_GGUF,
    "--outtype", "q8_0"
]

result = subprocess.run(convert_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("GGUF conversion failed.")

print(f"\n🎉 Zero-drift GGUF successfully created at:\n   {LOCAL_GGUF}")

⚙️ ZERO-DRIFT TOPO-CBP GGUF GENERATION
Loading zero-drift Topo-CBP weights...
Patching 1 tensors into model-00017-of-000017.safetensors...
✅ Shard patched successfully in safetensors format.
Running llama.cpp conversion...

🎉 Zero-drift GGUF successfully created at:
   /content/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf


## inference-local

In [2]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 113.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [19]:
from llama_cpp import Llama

# Load your newly generated zero-drift Topo-CBP governed Q8_0 GGUF model
llm = Llama(
    model_path="/content/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf",
    n_ctx=4096,
    n_threads=8,
    verbose=False,
    n_batch=512
)

# Run a test inference prompt
output = llm(
    "Explain the principles of Topological Continual Backpropagation.",
    max_tokens=1024,
    temperature=0.2,
    stop=["<|im_end|>"]
)

print("\n--- Model Output ---")
print(output['choices'][0]['text'])


--- Model Output ---
<think>
Okay, so I'm trying to understand Topological Continual Backpropagation. I remember that backpropagation is a key algorithm in training neural networks, using gradients to update weights. Continual learning, on the other hand, is about learning tasks one after another without forgetting previous ones. Topological Continual Backpropagation, or TCB, seems to combine these concepts but with a twist involving topology.

The user's explanation mentioned that TCB relies on the mathematical framework of algebraic topology. I'm a bit fuzzy on that, but I know it deals with properties preserved under continuous deformations, like connectedness and holes. So, how does that apply to neural networks?

They talked about the state space of a neural network, which I think is the set of all possible activation patterns across the network. The idea is that each task corresponds to a specific geometric structure in this space. So, when learning a new task, the network's sta

## hf

In [20]:
import os
import sys
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'
repo_id = f"{user_or_name}/deepseek-r1-distill-llama-70b-topo-governed-q8_0-gguf"
local_file_path = "/content/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"

print("=" * 70)
print("🚀 UPLOADING GGUF TO HUGGING FACE")
print("=" * 70)

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True
)

print(f"Uploading {local_file_path} to {repo_id}...")
api.upload_file(
    path_or_fileobj=local_file_path,
    path_in_repo="deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf",
    repo_id=repo_id,
    repo_type="model"
)

print(f"✅ Successfully uploaded to https://huggingface.co/{repo_id}")

🚀 UPLOADING GGUF TO HUGGING FACE
Uploading /content/deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf to frankmorales2020/deepseek-r1-distill-llama-70b-topo-governed-q8_0-gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b-topo-governed-q8_0.gguf:   0%|          |  524kB / 75.0GB            

Validating                    :           |   0%            

✅ Successfully uploaded to https://huggingface.co/frankmorales2020/deepseek-r1-distill-llama-70b-topo-governed-q8_0-gguf


## inference-hf

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --no-cache-dir

In [2]:
from llama_cpp import llama_supports_gpu_offload

# This will return True if compiled with CUDA/Metal support
print("GPU Offload Supported:", llama_supports_gpu_offload())

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 97250 MiB):
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes, VRAM: 97250 MiB


GPU Offload Supported: True


In [3]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Download the governed GGUF model directly from your Hugging Face repository
model_path = hf_hub_download(
    repo_id="frankmorales2020/deepseek-r1-distill-llama-70b-topo-governed-q8_0-gguf",
    filename="deepseek-r1-distill-llama-70b-topo-governed-q8_0.gguf"
)

# Initialize the model with llama-cpp-python
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,
    n_gpu_layers=-1,
    verbose=False
)

# Run inference
output = llm(
    "Explain the zero-drift properties of Topological Continual Backpropagation.",
    max_tokens=512,
    temperature=0.2,
    stop=["<|im_end|>"]
)

print("\n--- Model Output ---")
print(output['choices'][0]['text'])

deepseek-r1-distill-llama-70b-topo-gover(…): reconstructing file:   0%|          |  0.00B / 75.0GB            

deepseek-r1-distill-llama-70b-topo-gover(…): downloading bytes:           |  0.00B            


--- Model Output ---
<think>
Okay, so I'm trying to understand this Topological Continual Backpropagation (TCBP) and why it's supposed to have zero drift. I remember that in neural networks, especially in continuous learning or lifelong learning scenarios, a big problem is catastrophic forgetting. That's when the network forgets its previous tasks as it learns new ones. So, methods like EWC (Elastic Weight Consolidation) try to prevent this by making some weights harder to change if they're important for previous tasks. But I'm not exactly sure how TCBP works and why it's better in terms of drift.

First, I need to recall what backpropagation does. It's the process of adjusting the weights of the network by calculating the gradient of the loss with respect to the weights. In standard backpropagation, during each training step, the weights are updated in the direction that reduces the loss. But in a continuous learning setting, this can cause the weights to move away from their optimal

In [4]:
!nvidia-smi

Sun Nov  8 03:52:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   35C    P0             86W /  600W |   72927MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----